# 常見機器學習演算法

## 學習目標

本 Notebook 以「中級 AI 應用規劃師」考試常見概念為核心，透過小型資料集實作下列重點：

1. 理解監督式學習中迴歸任務的基本流程。
2. 比較線性迴歸、Ridge 迴歸、Lasso 迴歸與 SVR 的差異。
3. 使用 MSE、RMSE、MAE、R2 評估模型表現。
4. 觀察正規化如何影響模型係數與特徵選擇。
5. 建立選擇演算法時的實務判斷能力。


In [ ]:
# ── 環境設定 ────────────────────────────────────
# 載入本章節所需的 Python 套件，並確認 sklearn、numpy、pandas 與 matplotlib 可正常使用。

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

np.random.seed(42)
print('環境設定完成')


## 核心概念說明

### 監督式學習與迴歸任務

監督式學習會使用已知答案的資料進行訓練。若目標值是連續數值，例如房價、銷售額、用電量或風險分數，通常稱為迴歸任務。

### 線性迴歸

線性迴歸假設輸入特徵與目標值之間存在近似線性關係，模型會學習每個特徵的係數。係數越大，代表該特徵對預測結果的影響通常越明顯。

### Ridge 與 Lasso

Ridge 與 Lasso 都是在迴歸模型中加入正規化，目的是降低過度擬合風險。

- Ridge 使用 L2 正規化，會壓小係數，但通常不會讓係數變成 0。
- Lasso 使用 L1 正規化，可能讓部分係數變成 0，因此具備特徵選擇效果。

### SVR

支援向量迴歸 SVR 不像一般線性迴歸追求所有誤差都最小，而是允許預測落在 epsilon 範圍內。搭配 RBF kernel 時，可處理非線性資料。


In [ ]:
# ── 示範：建立迴歸資料與評估函式 ──────────────────────────
# 建立一組可重現的迴歸資料，並定義常用評估指標函式，包含 MSE、RMSE、MAE 與 R2。

import numpy as np
import pandas as pd
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

np.random.seed(42)

X, y = make_regression(
    n_samples=220,
    n_features=5,
    n_informative=3,
    noise=18,
    random_state=42
)

feature_names = [f'特徵_{i+1}' for i in range(X.shape[1])]
df = pd.DataFrame(X, columns=feature_names)
df['目標值'] = y

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

def evaluate_regression(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return pd.Series({'MSE': mse, 'RMSE': rmse, 'MAE': mae, 'R2': r2})

print(df.head())
print('\n訓練資料筆數:', X_train.shape[0])
print('測試資料筆數:', X_test.shape[0])


In [ ]:
# ── 示範：線性迴歸模型 ───────────────────────────────
# 訓練線性迴歸模型，觀察模型評估結果與各特徵係數，理解可解釋性在迴歸任務中的角色。

import numpy as np
import pandas as pd
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

np.random.seed(42)

X, y = make_regression(
    n_samples=220,
    n_features=5,
    n_informative=3,
    noise=18,
    random_state=42
)
feature_names = [f'特徵_{i+1}' for i in range(X.shape[1])]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

def evaluate_regression(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    return pd.Series({
        'MSE': mse,
        'RMSE': np.sqrt(mse),
        'MAE': mean_absolute_error(y_true, y_pred),
        'R2': r2_score(y_true, y_pred)
    })

model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

metrics = evaluate_regression(y_test, y_pred)
coef_table = pd.DataFrame({
    '特徵': feature_names,
    '係數': model.coef_
}).sort_values('係數', key=np.abs, ascending=False)

print('線性迴歸評估結果')
print(metrics.round(3))
print('\n特徵係數')
print(coef_table.round(3))


In [ ]:
# ── 示範：Ridge 與 Lasso 正規化比較 ──────────────────
# 比較線性迴歸、Ridge 與 Lasso 的評估結果與係數差異，觀察 Lasso 可能將不重要特徵係數壓縮為 0。

import numpy as np
import pandas as pd
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

np.random.seed(42)

X, y = make_regression(
    n_samples=240,
    n_features=8,
    n_informative=3,
    noise=25,
    random_state=42
)
feature_names = [f'特徵_{i+1}' for i in range(X.shape[1])]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

def evaluate_regression(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    return {
        'MSE': mse,
        'RMSE': np.sqrt(mse),
        'MAE': mean_absolute_error(y_true, y_pred),
        'R2': r2_score(y_true, y_pred)
    }

models = {
    'LinearRegression': make_pipeline(StandardScaler(), LinearRegression()),
    'Ridge alpha=10': make_pipeline(StandardScaler(), Ridge(alpha=10)),
    'Lasso alpha=5': make_pipeline(StandardScaler(), Lasso(alpha=5, max_iter=10000))
}

rows = []
coefs = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    rows.append({'模型': name, **evaluate_regression(y_test, pred)})
    coefs[name] = model.named_steps[list(model.named_steps.keys())[-1]].coef_

result = pd.DataFrame(rows).set_index('模型')
coef_table = pd.DataFrame(coefs, index=feature_names)

print('模型評估比較')
print(result.round(3))
print('\n係數比較')
print(coef_table.round(3))
print('\nLasso 係數為 0 的特徵數:', int((np.abs(coef_table['Lasso alpha=5']) < 1e-8).sum()))


In [ ]:
# ── 實際應用：非線性資料上的 SVR ────────────────────────
# 建立一組非線性資料，比較線性迴歸與 RBF kernel SVR 的表現，理解 SVR 適合處理非線性趨勢的情境。

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

np.random.seed(42)

X = np.linspace(-3, 3, 180).reshape(-1, 1)
y = 12 * np.sin(X[:, 0]) + 0.8 * X[:, 0] ** 2 + np.random.normal(0, 1.2, size=X.shape[0])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

def evaluate_regression(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    return {
        'MSE': mse,
        'RMSE': np.sqrt(mse),
        'MAE': mean_absolute_error(y_true, y_pred),
        'R2': r2_score(y_true, y_pred)
    }

models = {
    'LinearRegression': make_pipeline(StandardScaler(), LinearRegression()),
    'SVR RBF kernel': make_pipeline(StandardScaler(), SVR(kernel='rbf', C=30, gamma='scale', epsilon=0.5))
}

rows = []
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    rows.append({'模型': name, **evaluate_regression(y_test, pred)})

result = pd.DataFrame(rows).set_index('模型')
print('非線性資料模型評估比較')
print(result.round(3))

X_plot = np.linspace(X.min(), X.max(), 300).reshape(-1, 1)
plt.figure(figsize=(8, 5))
plt.scatter(X_train[:, 0], y_train, alpha=0.45, label='訓練資料')
plt.scatter(X_test[:, 0], y_test, alpha=0.75, label='測試資料')

for name, model in models.items():
    y_plot = model.predict(X_plot)
    plt.plot(X_plot[:, 0], y_plot, linewidth=2, label=name)

plt.title('線性迴歸與 RBF SVR 在非線性資料上的比較')
plt.xlabel('特徵 X')
plt.ylabel('目標值 y')
plt.legend()
plt.grid(alpha=0.3)
plt.show()
